# NuNo vs RPT on Kaggle 2×T4
Notebook này chạy **cùng training pipeline** cho NuNo và RPT. Mặc định dùng Qwen2.5-0.5B student + Qwen2.5-3B teacher để vừa VRAM T4. Hãy bật `Accelerator: GPU T4 x2` và Internet trong Kaggle Settings.

In [ ]:
!nvidia-smi --query-gpu=index,name,memory.total --format=csv
import torch
assert torch.cuda.device_count() == 2, f'Notebook cần 2 GPU, hiện có {torch.cuda.device_count()}'
print('CUDA:', torch.version.cuda, '| GPUs:', torch.cuda.device_count())

In [ ]:
%pip install -q transformers==4.57.3 peft==0.18.1 datasets deepspeed rouge-score numerize rich wandb celery nltk

In [ ]:
from pathlib import Path
import os, subprocess
WORKDIR = Path('/kaggle/working/nuno-kd')
if not WORKDIR.exists():
    subprocess.run(['git', 'clone', 'https://github.com/chiiipk/nuno-kd.git', str(WORKDIR)], check=True)
else:
    subprocess.run(['git', '-C', str(WORKDIR), 'pull', '--ff-only'], check=True)
os.chdir(WORKDIR)
print('Commit:', subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip())

In [ ]:
# Smoke test trước khi tải model/train
subprocess.run(['python3', '-m', 'unittest', 'discover', '-s', 'tests', '-v'], check=True)

## Data
Repo local đã có hai file Qwen/Gemma, mỗi file 79.751 mẫu với schema `prompt` + `generated_text`. Vì `data/` bị Git ignore, trên Kaggle hãy upload thư mục `data` thành một Kaggle Dataset rồi **Add Input**. Cell dưới sẽ tự tìm file Qwen trong `/kaggle/input`; không cần gõ tên dataset.

In [ ]:
# Tự tìm data: local trước, sau đó mọi Kaggle Dataset đã Add Input.
local_qwen = WORKDIR / 'data/dpo/Qwen/Qwen2.5-14B-Instruct/generated_train.jsonl'
kaggle_files = sorted(Path('/kaggle/input').rglob('generated_train.jsonl'))
qwen_files = [p for p in kaggle_files if 'Qwen' in str(p) or 'qwen' in str(p)]
RAW_JSONL = local_qwen if local_qwen.exists() else (qwen_files or kaggle_files or [None])[0]
STUDENT = 'Qwen/Qwen2.5-0.5B-Instruct'
TEACHER = 'Qwen/Qwen2.5-3B-Instruct'
TRAIN_SAMPLES = 5000       # smoke run: 200; full run: -1
EPOCHS = 1
MAX_LENGTH = 512           # tăng lên 768/1024 nếu còn VRAM
NNM_RATIO = 0.1
SEED = 10
assert RAW_JSONL is not None and RAW_JSONL.exists(), (
    'Không tìm thấy generated_train.jsonl. Hãy Add Input dataset chứa folder data.'
)
print('Using data:', RAW_JSONL, '| size MB:', round(RAW_JSONL.stat().st_size / 2**20, 1))

In [ ]:
# Preprocess một lần; NuNo và RPT dùng chung output này.
PROCESSED_ROOT = Path('/kaggle/working/processed_data')
DATA_DIR = PROCESSED_ROOT / TEACHER
if not (DATA_DIR / 'train_0.idx').exists():
    cmd = [
        'python3', 'tools/process_data_ultraInteract.py',
        '--data-dir', str(RAW_JSONL),
        '--processed-data-dir', str(PROCESSED_ROOT),
        '--model-path', TEACHER,
        '--data-process-workers', '2',
        '--max-prompt-length', '384',
        '--max-length', str(MAX_LENGTH),
        '--dev-num', '100', '--only-prompt', '--model-type', 'qwen',
    ]
    subprocess.run(cmd, check=True, env={**os.environ, 'PYTHONPATH': '.'})
print('Processed data:', DATA_DIR)

In [ ]:
def run_experiment(variant: str, train_samples: int = TRAIN_SAMPLES):
    assert variant in {'nuno', 'rpt'}
    save_dir = Path('/kaggle/working/results') / f'{variant}_qwen05b_from_3b'
    args = [
        'torchrun', '--standalone', '--nproc_per_node=2', 'finetune.py',
        '--base-path', '.', '--model-path', STUDENT, '--teacher-model-path', TEACHER,
        '--ckpt-name', 'qwen2.5-0.5B-it', '--teacher-ckpt-name', 'qwen2.5-3B-it',
        '--teacher-model-fp16', '--n-gpu', '2', '--data-dir', str(DATA_DIR),
        '--num-workers', '2', '--dev-num', '-1', '--train-num', str(train_samples),
        '--lr', '1e-4', '--batch-size', '1', '--eval-batch-size', '1',
        '--gradient-accumulation-steps', '8', '--gradient-checkpointing',
        '--warmup-iters', '0', '--lr-decay-style', 'cosine', '--weight-decay', '1e-2',
        '--clip-grad', '1.0', '--epochs', str(EPOCHS), '--kd-ratio', '1.0',
        '--max-length', str(MAX_LENGTH), '--max-prompt-length', '384',
        '--do-train', '--save-interval', '-1', '--eval-interval', '-1',
        '--log-interval', '10', '--mid-log-num', '-1', '--save', str(save_dir),
        '--seed', str(SEED), '--deepspeed',
        '--deepspeed_config', 'configs/deepspeed/ds_config_zero2_offload.json',
        '--type', 'adaptive-sfkl', '--skew-alpha', '0.1', '--do-sample',
        '--top-k', '0', '--top-p', '1.0', '--temperature', '1.0',
        '--student-gen', '--gen-num-beams', '1', '--gen-top-p', '1.0',
        '--init-threshold', '0.0', '--loss-eps', '0.1', '--capacity', '200',
        '--replay-ratio', 'decreasing', '--mixed-alpha', '0.5',
        '--peft', 'lora', '--peft-lora-r', '16', '--peft-lora-alpha', '32',
        '--nnm', '--loss-variant', variant, '--nnm-ratio', str(NNM_RATIO),
        '--nnm-K', '64', '--nnm-n-layers', '4', '--nnm-d-prime', '128',
        '--nnm-centroid-batches', '100', '--nnm-eta', '0.05', '--nnm-T-dead', '50',
        '--nnm-ns-iters', '5', '--nnm-warmup-steps', '20', '--nnm-ramp-steps', '50',
        '--delta-threshold', '0.03',
    ]
    env = {**os.environ, 'PYTHONPATH': '.', 'WANDB_DISABLED': 'true',
           'TOKENIZERS_PARALLELISM': 'false', 'NCCL_P2P_DISABLE': '1'}
    print('Running:', variant, '| output:', save_dir)
    subprocess.run(args, check=True, env=env)
    return save_dir

## Chạy thử rồi chạy thí nghiệm
Đầu tiên chạy **một** variant với 200 mẫu. Nếu không OOM, đổi sang `TRAIN_SAMPLES`. Để so sánh công bằng, NuNo và RPT phải dùng cùng `TRAIN_SAMPLES`, seed và mọi config khác.

In [ ]:
# Smoke run ngắn
run_experiment('rpt', train_samples=200)

In [ ]:
# Full comparison: chạy tuần tự để mỗi run dùng cả 2 GPU.
# nuno_dir = run_experiment('nuno')
# rpt_dir = run_experiment('rpt')